# 15 — Official test submission by registry `model_id` (explicit `max_new_tokens`)

**Summary:** Loads one adapter from `<WORKFLOW_ROOT>/models/model_registry.csv`, reads **`data/raw/test.csv`**, generates with a **single** `max_new_tokens` you set (not from registry metadata), applies **`POSTPROCESS_METHOD`**, and writes **`outputs/submissions/<model_id>_submission.csv`** (filesystem-safe stem derived from `MODEL_ID`). With **`N_SANITY`** set, only that many randomly sampled test rows are generated and written.

**Parameters:** `RUN_PROFILE_ID` (match notebook **03**), `MODEL_ID`, `max_new_tokens`, `TEST_CSV` (default raw test), `BASE_MODEL_ID` (optional; else from registry `training_config_json`), `POSTPROCESS_METHOD`, `N_SANITY` (optional; if set, a random subset of that many test rows, `random_state=42`).

#### Colab / install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install transformers peft accelerate bitsandbytes pandas tqdm lxml

#### Parameters + run

In [ ]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RUN_PROFILE_ID = 'run_1'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
MODELS_ROOT = WORKFLOW_ROOT / 'models'
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
SUBMISSIONS_DIR = OUTPUTS_DIR / 'submissions'

MODEL_ID = 'REPLACE_WITH_REGISTRY_MODEL_ID'
BASE_MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'  # or e.g. 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
max_new_tokens = 1024
TEST_CSV = RAW_DIR / 'test.csv'
POSTPROCESS_METHOD = 'current_default_sanitizer'
N_SANITY = None  # e.g. 10 for a quick random subset; None = all test rows

def submission_filename_stem(model_id: str) -> str:
    """Filesystem-safe stem; keeps letters, digits, dot, hyphen, underscore."""
    s = str(model_id).strip()
    s = re.sub(r'[^\w.-]+', '_', s, flags=re.UNICODE)
    s = s.strip('._') or 'model'
    return s


from src.core.dataframe import choose_first_existing
from src.eval.postprocess_presets import get_postprocess_fn
from src.inference.generation import generate_svg_raw_prediction
from src.training.lora.modeling import load_inference_adapter
from src.training.lora.registry import load_registry, resolve_adapter_path
from src.training.prompts import format_svg_instruction_example

if not MODEL_ID or MODEL_ID == 'REPLACE_WITH_REGISTRY_MODEL_ID':
    raise ValueError('Set MODEL_ID to your registry model_id string.')

test_df = pd.read_csv(TEST_CSV)
if 'id' not in test_df.columns:
    raise ValueError('test CSV needs id column')
if N_SANITY is not None:
    n_want = int(N_SANITY)
    if n_want <= 0:
        raise ValueError('N_SANITY must be a positive integer')
    n = min(n_want, len(test_df))
    if n < n_want:
        print(f'N_SANITY={n_want} exceeds test size {len(test_df)}; using n={n}')
    test_df = test_df.sample(n=n, random_state=42).reset_index(drop=True)
    print(f'N_SANITY subset: {n} random test rows (random_state=42)')
prompt_col = choose_first_existing(test_df, ['prompt', 'description', 'text'], 'test_df')

reg = load_registry(PROJECT_DIR, models_root=MODELS_ROOT)
rrow = reg[reg['model_id'].astype(str) == str(MODEL_ID)]
if len(rrow) != 1:
    raise ValueError(f'model_id not unique in registry: {MODEL_ID!r}')
r0 = rrow.iloc[0]
adapter = resolve_adapter_path(PROJECT_DIR, str(r0['adapter_dir']))

resolved_base = BASE_MODEL_ID
if resolved_base is None:
    cfg = json.loads(r0['training_config_json'])
    resolved_base = cfg.get('base_model_id') or cfg.get('model_id')
if not resolved_base:
    raise ValueError('Set BASE_MODEL_ID explicitly (not found in registry training_config_json).')

mxt = int(max_new_tokens)
post_fn = get_postprocess_fn(POSTPROCESS_METHOD)
tokenizer, model = load_inference_adapter(adapter, resolved_base)

rows = []
for _, trow in tqdm(test_df.iterrows(), total=len(test_df)):
    pid = trow['id']
    prompt_text = str(trow[prompt_col])
    full_prompt = format_svg_instruction_example(prompt_text, svg_text=None, include_answer=False)
    raw = generate_svg_raw_prediction(full_prompt, tokenizer, model, max_new_tokens=mxt)
    svg = post_fn(raw)
    rows.append({'id': pid, 'svg': svg})

del model, tokenizer

sub = pd.DataFrame(rows, columns=['id', 'svg'])
stem = submission_filename_stem(MODEL_ID)
out_path = SUBMISSIONS_DIR / f'{stem}_submission.csv'
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
sub.to_csv(out_path, index=False)
print('Wrote', out_path, sub.shape, 'max_new_tokens=', mxt)